# 1. Train MACEField on BaTiO₃

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/01_train_batio3.ipynb)

This short from-scratch exercise trains on a five-atom BaTiO₃-like cell. The bundled teaching set varies the Ti displacement inside its oxygen cage and the z-directed electric field. Its energy, forces, and polarization labels come from an analytic Landau surface, not DFT or experiment.

This notebook writes its own model under a new run directory. Later lessons use a separate bundled checkpoint, so they do not depend on running this training cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## Check the training data

The fixed split contains 345 training and 132 validation frames. Fields span both signs, and the displacement moves Ti along z relative to the oxygen cage.


In [ ]:
import numpy as np
from ase.io import read

TRAIN = DATA_DIR / "batio3/macefield_batio3_toy_train.extxyz"
VALID = DATA_DIR / "batio3/macefield_batio3_toy_valid.extxyz"
train = read(TRAIN, index=":")
valid = read(VALID, index=":")
print("train / validation frames:", len(train), "/", len(valid))
print("atoms per frame:", sorted({len(a) for a in train + valid}))
print("train label keys:", sorted(train[0].info))
print("field-z range (V/Å):",
      min(a.info["REF_electric_field"][2] for a in train),
      max(a.info["REF_electric_field"][2] for a in train))
print("displacement range (Å):",
      min(a.info["REF_mode_displacement"] for a in train),
      max(a.info["REF_mode_displacement"] for a in train))


## Train a compact model

The defaults make a short Colab demonstration. Increase the epoch count and assess independent data before treating a model as useful. BECs are not training labels in this toy set; they can still be computed as derivatives of the trained energy in the next lesson.


In [ ]:
import os
import time
import yaml
import torch
import subprocess
import sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_EPOCHS = 30 if DEVICE == "cuda" else 3
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", DEFAULT_EPOCHS))
RUN_DIR = TUTORIAL_DIR / "runs" / f"batio3_from_scratch_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = {
    "name": "MACEField-BaTiO3-toy",
    "model": "MACEField",
    "loss": "universal_field",
    "train_file": str(TRAIN),
    "valid_file": str(VALID),
    "valid_fraction": 0,
    "E0s": {8: 0.0, 22: 0.0, 56: 0.0},
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "electric_field_key": "REF_electric_field",
    "polarization_key": "REF_polarization",
    "compute_forces": True,
    "compute_stress": False,
    "compute_polarization": True,
    "compute_becs": False,
    "compute_polarizability": False,
    "energy_weight": 1.0,
    "forces_weight": 100.0,
    "stress_weight": 0.0,
    "polarization_weight": 0.01,
    "becs_weight": 0.0,
    "polarizability_weight": 0.0,
    "hidden_irreps": "16x0e + 16x1o",
    "num_interactions": 2,
    "max_L": 1,
    "r_max": 5.0,
    "batch_size": 32,
    "valid_batch_size": 32,
    "num_workers": 0,
    "lr": 0.005,
    "max_num_epochs": EPOCHS,
    "patience": max(10, EPOCHS),
    "eval_interval": 1,
    "seed": 23,
    "default_dtype": "float32",
    "device": DEVICE,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG = RUN_DIR / "train.yml"
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print("device:", DEVICE, "| epochs:", EPOCHS)
print("configuration:", CONFIG)
print("output model directory:", config["model_dir"])

subprocess.run(
    [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
    cwd=PACKAGE_REPO,
    check=True,
)


## What this model learned

The model saw Ti displacements and fields of both signs, which makes it a useful compact exercise in field-conditioned training. The labels are synthetic by design. Do not use this model to report quantitative BaTiO₃ properties or compare directly with an experimental hysteresis loop.
